In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import h5py 
import sys
import data_analysis.mea_analysis as mea
from tqdm import tqdm
from pathlib import Path
import json
import cv2

In [2]:
BRW_BASE_FOLDER = Path("/media/ferdinand-forberger/RawDataBackUp")
MAIN_FOLDER = Path("/media/ferdinand-forberger/Seagate Portable Drive")
GRAPH_METRICS_PATH = MAIN_FOLDER / "graph_metrics.xlsx"



df_overview = mea.get_overview(MAIN_FOLDER)
df_overview = df_overview[df_overview["sles_saved"]==True]
df_overview.reset_index(inplace=True, drop=True)
df_overview["image_path"]= df_overview["folder_path"].astype(str) + "/org_image.png"
df_overview["mask_path"] = df_overview["folder_path"].astype(str) + "/mask.png"

#get metadata file to have info about which slice belongs to which group
df_metadata = pd.read_excel("/media/ferdinand-forberger/RawDataBackUp/meta_data_file.xlsx")
df_metadata["folder"] = df_metadata["animal_number"].astype(str) + "_" + df_metadata["slice_number"].astype(str)
df_overview = pd.merge(df_overview, df_metadata, left_on="folder", right_on="folder", how="left")
df_overview["sttc_path"] = df_overview["folder_path"] / "sttc.npy"

In [7]:
aligned_maps = {"Freq" : [],
                 "Ampl" : []}

for i, row in tqdm(df_overview.iterrows(), total=len(df_overview)):
    json_path = row["json_path"]
    st_df, st, clu, best_cha_st, vectorized_map, amplitudes= mea.get_stdf(parent_folder_path=row["folder_path"],
                                                                    return_amplitudes=True)
    st_df = st_df.loc[st_df["num_spikes"] > 100]
    st_df["freq"] = st_df["num_spikes"] / 600
    st_df["amplitude"] = st_df["clu"].apply(lambda u: unit_to_amplitude(u, clu, amplitudes))
    st_df_temp = st_df.drop_duplicates(subset=["chan_best"], keep="first")


    org_img = plt.imread(row["image_path"])
    chan_map = np.arange(4096).reshape(64,64)
    chan_map_rotated = transform_matrix_from_json(chan_map, 
                                                json_path, 
                                                original_image=org_img)


    freq_map = np.zeros(4096)
    for _, sub_row in st_df_temp.iterrows():
        chan = sub_row["chan_best"]
        freq_map[chan] = sub_row["freq"]
    freq_map = freq_map.reshape(64,64)

    ampl_map = np.zeros(4096)
    for _, sub_row in st_df_temp.iterrows():
        chan = sub_row["chan_best"]
        ampl_map[chan] = sub_row["amplitude"]
    ampl_map = ampl_map.reshape(64,64)

    freq_map_rotated = transform_matrix_from_json(freq_map, 
                                                json_path, 
                                                original_image=org_img)
    ampl_map_rotated = transform_matrix_from_json(ampl_map, 
                                                json_path, 
                                                original_image=org_img)
    
    freq_map_rotated[freq_map_rotated==0] = np.nan
    ampl_map_rotated[ampl_map_rotated==0] = np.nan

    aligned_maps["Freq"].append(freq_map_rotated)
    aligned_maps["Ampl"].append(ampl_map_rotated)
    
aligned_maps["Freq"] = np.array(aligned_maps["Freq"])
aligned_maps["Ampl"] = np.array(aligned_maps["Ampl"])

  0%|          | 0/22 [00:00<?, ?it/s]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 408, disagreement: 0
Number of good clusters (Original): 121
Number of good clusters (Changed): 121
-------------------------------------------------------


  5%|▍         | 1/22 [00:00<00:12,  1.75it/s]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 554, disagreement: 0
Number of good clusters (Original): 119
Number of good clusters (Changed): 119
-------------------------------------------------------


  9%|▉         | 2/22 [00:02<00:26,  1.33s/it]

...

 91%|█████████ | 20/22 [01:24<00:08,  4.34s/it]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 450, disagreement: 0
Number of good clusters (Original): 91
Number of good clusters (Changed): 91
-------------------------------------------------------


 95%|█████████▌| 21/22 [01:28<00:04,  4.12s/it]

Comparison of Labels (Original vs. Changed)
KSLabel agreement: 323, disagreement: 0
Number of good clusters (Original): 55
Number of good clusters (Changed): 55
-------------------------------------------------------


100%|██████████| 22/22 [01:30<00:00,  4.13s/it]


In [8]:
save_folder = Path("data")

np.save(save_folder / "aligned_freq_maps.npy", aligned_maps["Freq"])
np.save(save_folder / "aligned_ampl_maps.npy", aligned_maps["Ampl"])